# 07 — YOLO training

Trains YOLOv8n from the cached COCO checkpoint, validates on `valid`, saves the
best weights and logs the run to MLflow.

Only the hyperparameters in `YoloConfig` are exposed. Device-specific defaults
(CPU: 10 epochs / 320 px / batch 8, GPU: 100 epochs / 640 px / batch 32) are
selected automatically. For a quick smoke run, uncomment the small overrides.

## 1. Train and log the run

The call below runs training and validation. `log_training` records the
hyperparameters, per-epoch curves, validation metrics, artefacts and the model
in the `yolo-brain-tumor` experiment.

In [ ]:
from IPython.display import Image, display

from diagnostic_des_tumeurs.config import Paths, YOLO_EXPERIMENT, YoloConfig
from diagnostic_des_tumeurs.yolo.functions.tracking import log_training
from diagnostic_des_tumeurs.yolo.functions.training import train

paths = Paths.from_root()
config = YoloConfig()
# config = YoloConfig(epochs=1, imgsz=160, batch=8, patience=1, fraction=0.02, device="cpu")

result = train(paths, config)
run_id = log_training(result, config, paths, YOLO_EXPERIMENT)
print("MLflow run:", run_id)
print("Weights   :", result.weights)
result.validation_metrics

## 2. Per-class validation metrics

`mAP50` is the metric at IoU 0.5; `mAP50-95` averages IoU thresholds from 0.5
to 0.95.

In [ ]:
result.per_class

## 3. Training curves

The first image is Ultralytics' full results figure; the second one is the
loss/metric summary produced by the package.

In [ ]:
display(Image(filename=str(result.figure_paths["results"])))
display(Image(filename=str(result.figure_paths["curves"])))

## 4. Saved artefacts

`training_summary.json` contains the metrics and hyperparameters needed to
reuse the model; the weights are copied to a stable path
(`models/yolo/brain_tumor_yolov8n.pt`).

In [ ]:
for path in (result.weights, paths.yolo_training_summary, paths.yolo_training_args, paths.yolo_training_results):
    print(path, "exists:", path.exists())